In [3]:
import os
import subprocess
from IPython.display import display, Markdown

class CommandSimulator:
    def __init__(self):
        self.cwd = os.getcwd()  # 当前工作目录
        self.command_map = {
            'cd': self._change_dir,
            'mkdir': self._make_dir,
            'ls': self._list_dir,
            'pwd': self._print_work_dir,
        }
    
    def run(self):
        display(Markdown("**模拟命令行** (输入 `exit` 退出)"))
        while True:
            try:
                cmd = input(f"({self.cwd}) $ ").strip()
                if cmd.lower() in ('exit', 'quit'):
                    break
                self._execute(cmd)
            except KeyboardInterrupt:
                break
    
    def _execute(self, cmd):
        """解析并执行命令"""
        parts = cmd.split()
        if not parts:
            return
            
        cmd_name = parts[0]
        args = parts[1:]
        
        # 处理内置命令
        if cmd_name in self.command_map:
            self.command_map[cmd_name](*args)
        # 执行系统命令
        else:
            self._run_system_command(cmd)
    
    # ---- 内置命令实现 ----
    def _change_dir(self, *args):
        """模拟 cd 命令"""
        if not args:
            target_dir = os.path.expanduser("~")  # 默认回家目录
        else:
            target_dir = args[0]
        
        try:
            new_dir = os.path.abspath(os.path.join(self.cwd, target_dir))
            if os.path.isdir(new_dir):
                self.cwd = new_dir
            else:
                display(Markdown(f"**错误:** 目录不存在 `{new_dir}`"))
        except Exception as e:
            display(Markdown(f"**错误:** `{str(e)}`"))
    
    def _make_dir(self, *args):
        """模拟 mkdir 命令"""
        for dirname in args:
            try:
                full_path = os.path.join(self.cwd, dirname)
                os.makedirs(full_path, exist_ok=True)
                display(Markdown(f"创建目录: `{full_path}`"))
            except Exception as e:
                display(Markdown(f"**错误:** `{str(e)}`"))
    
    def _list_dir(self, *args):
        """模拟 ls 命令"""
        try:
            items = os.listdir(self.cwd)
            display(Markdown("```\n" + "\n".join(items) + "\n```"))
        except Exception as e:
            display(Markdown(f"**错误:** `{str(e)}`"))
    
    def _print_work_dir(self, *args):
        """模拟 pwd 命令"""
        display(Markdown(f"```\n{self.cwd}\n```"))
    
    # ---- 系统命令 ----
    def _run_system_command(self, cmd):
        """执行非内置命令"""
        try:
            result = subprocess.run(
                cmd,
                shell=True,
                cwd=self.cwd,  # 关键：在模拟的当前目录执行
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True
            )
            display(Markdown(f"```\n{result.stdout}\n```"))
        except Exception as e:
            display(Markdown(f"**错误:** `{str(e)}`"))

# 启动模拟器
CommandSimulator().run()

**模拟命令行** (输入 `exit` 退出)

**错误:** 目录不存在 `/mnt/disk_0/IC/python/google/~`

```
/bin/sh: ll: command not found

```

```
lib32
snap
sys
tmp
swap.img
dev
root
usr
bin
boot
.Recycle_bin
libx32
opt
sbin
mnt
etc
lost+found
var
patch
run
www
lib
proc
lib64
home
media
srv
```

```
/bin/sh: ll: command not found

```

```
dy
www
```

In [4]:
from IPython.display import HTML

class AdvancedShellSimulator(CommandSimulator):
    def __init__(self):
        super().__init__()
        self.env = {"USER": "jupyter_user"}  # 模拟环境变量
        self.command_map.update({
            'export': self._set_env,
            'echo': self._echo,
            'grep': self._grep,
        })
    def _change_dir(self, *args):
        """改进版 cd 命令，支持 ~ 和相对路径"""
        if not args:
            target_dir = os.path.expanduser("~")  # 默认切换到用户家目录
        else:
            target_dir = args[0]
        
        try:
            # 特殊处理 ~
            if target_dir.startswith("~"):
                new_dir = os.path.expanduser(target_dir)
            # 相对路径处理
            else:
                new_dir = os.path.abspath(os.path.join(self.cwd, target_dir))
            
            if os.path.isdir(new_dir):
                self.cwd = new_dir
            else:
                display(Markdown(f"**错误:** 目录不存在 `{new_dir}`"))
        except Exception as e:
            display(Markdown(f"**错误:** `{str(e)}`"))
    def _set_env(self, *args):
        """模拟 export VAR=value"""
        if '=' in args[0]:
            var, value = args[0].split('=', 1)
            self.env[var] = value
    
    def _echo(self, *args):
        """模拟 echo 命令（支持环境变量）"""
        text = ' '.join(args)
        for var in self.env:
            text = text.replace(f'${var}', self.env[var])
        display(HTML(f"<pre>{text}</pre>"))
    
    def _grep(self, *args):
        """简易 grep 模拟"""
        if len(args) < 2:
            display(Markdown("**用法:** `grep pattern file`"))
            return
        
        pattern = args[0]
        filename = args[1]
        try:
            with open(os.path.join(self.cwd, filename)) as f:
                lines = [line for line in f if pattern in line]
                display(Markdown(f"```\n{''.join(lines)}\n```"))
        except Exception as e:
            display(Markdown(f"**错误:** `{str(e)}`"))

# 使用方式（支持管道等高级功能）
AdvancedShellSimulator().run()

**模拟命令行** (输入 `exit` 退出)

```
/bin/sh: ll: command not found

```

**错误:** 目录不存在 `/mnt/disk_0/IC/python/google/~`

```
/mnt/disk_0/IC/python/google
```

**错误:** 目录不存在 `/mnt/disk_0/IC/python/google/~`

...

```
lib32
snap
sys
tmp
swap.img
dev
root
usr
bin
boot
.Recycle_bin
libx32
opt
sbin
mnt
etc
lost+found
var
patch
run
www
lib
proc
lib64
home
media
srv
```

```
dy
www
```

```
.Xauthority
stack.info.290186
.profile
.viminfo
.pki
.condarc
.vscode-server
novas.conf
Documents
Music
.ssh
.local
.bashrc
.dmrc
verdiLog
Videos
.ubuntuserver2004.db
.xorgxrdp.10.log
.metals
.verdi_onesearch_history.log
Desktop
.sbt
.xorgxrdp.11.log
.ICEauthority
IC
.python_history
.dotnet
.gnupg
.teroshdl2_prj.json
Downloads
Public
Templates
.gitconfig
Pictures
.snpsinstaller
.xorgxrdp.10.log.old
simulation
.xsession
.conda
.sudo_as_admin_successful
.xorgxrdp.11.log.old
.ipython
softwares
.wget-hsts
csrc
.bash_logout
myenv
.lesshst
novas.rc
.ubuntuserver2004_27050.txt
.bash_history
stack.info.290253
.cache
.config
stack.info.391110
.ivy2
thinclient_drives
.xsession-errors
```

```
verdiLog
cpp
bin2hex.py
makefile
Test
makefile_test
uvm_senior
.gitignore
setup
.vscode
basic
Softmax
Instructions.txt
gitremote
.git
sva
python
uvm_lib
README.md
iveriloginst.txt
C
```

**错误:** 目录不存在 `/home/dy/IC/~`